# Ridge Regression on Geospatial Embeddings

Compares three types of location-based feature representations for predicting geospatial properties via ridge regression:

| Method | Type | Ridge-Interpretable? |
|---|---|---|
| **Earth embeddings** (GeoCLIP / SatCLIP) | Location embeddings | No (not really) |
| **SpLiCE** | Sparse text-based concepts | Yes --> top concepts |
| **SAE** | Sparse autoencoder concepts | Yes --> top concepts|

In [2]:
import os, torch, numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold
from eval import top_concepts

## Configuration

Set the dataset, embedding models, and hyperparameters here before running the sections below.

In [ ]:
from pathlib import Path
DATASET = "elevation"
DATASET_ROOT = "/data/mosaiks/elevation"
DATASET_PATH = Path(DATASET_ROOT)

Location encoder for earth embeddings

In [ ]:
from earth_embeddings import load_or_generate as load_location_embeddings

LOC_ENCODER = "satclip"

SpLiCE for sparse embeddings

In [ ]:
from splice_embeddings import load_or_generate as load_splice_embeddings

SPLICE_MODEL = "satclip_geospatial"
SPLICE_MODEL_STR = f"splice_{SPLICE_MODEL}_latlon"

SAE for sparse embeddings

In [ ]:
# from sparse_embeddings import load_or_generate as load_sparse_embeddings
# SAE_MODEL_PATH = todo

In [7]:
K = 10 # for top k concepts

In [8]:
SEED=42

## 1. Earth Embeddings (GeoCLIP / SatCLIP)

Location encoders from pretrained encoder -- serving as a baseline.

In [9]:
X_train, y_train = load_location_embeddings(LOC_ENCODER, DATASET, 'train')
X_test, y_test = load_location_embeddings(LOC_ENCODER, DATASET, 'test')

Loading cached embeddings from /data/mosaiks/elevation/elevation_geoclip_train_embeddings.pt
Loading cached embeddings from /data/mosaiks/elevation/elevation_geoclip_test_embeddings.pt


In [10]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
model = RidgeCV(alphas=[ 0.01, 0.1, 1.0, 10.0, 100.0], cv=cv) # can also do Ridge(alpha)
model.fit(X_train, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","[0.01, 0.1, ...]"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",KFold(n_split... shuffle=True)
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [11]:
model.alpha_

np.float64(0.1)

In [12]:
r2_score = model.score(X_test, y_test)
print(f"R2 on test set: {r2_score:.4f}")

R2 on test set: 0.6068


## 2. SpLiCE Embeddings

Here, each dimension corresponds to a named geographic concept, so the top-K ridge coefficients are directly interpretable.

In [13]:
X_train, y_train = load_splice_embeddings(SPLICE_MODEL, DATASET, 'train')
X_test, y_test = load_splice_embeddings(SPLICE_MODEL, DATASET, 'test')

Loading cached embeddings from /data/mosaiks/elevation/elevation_splice_satclip_geospatial_latlon_train_embeddings.pt
Loading cached embeddings from /data/mosaiks/elevation/elevation_splice_satclip_geospatial_latlon_test_embeddings.pt


In [14]:
model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]) # can also do Ridge(alpha)
model.fit(X_train, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","[0.01, 0.1, ...]"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",None
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [15]:
r2_score = model.score(X_test, y_test)
print(f"R2 on test set: {r2_score:.4f}")

R2 on test set: 0.6889


In [16]:
from models.splice_model import _splice_cfg
root, models = _splice_cfg()
concepts = torch.load(os.path.join(root, models[SPLICE_MODEL]["concepts"]), weights_only=False)
print(top_concepts(model, k=K, concepts=concepts))

['butte', 'basin', 'saltmarsh', 'cliff', 'saltpan', 'dam', 'moraine', 'gorge', 'slope', 'ridge']


Questions to ask here:
1. Do the top-k concepts make sense?
2. Is are performance similar to, better, or worse than the model performance on the original earth embeddings?

## 3. SAE Embeddings *(in progress)*

Set `SAE_MODEL_PATH` to checkpoint and uncomment the cells

In [ ]:
# X_train, y_train = load_sparse_embeddings(SPLICE_MODEL, DATASET, 'train')
# X_test, y_test = load_sparse_embeddings(SPLICE_MODEL, DATASET, 'test')

In [ ]:
# model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]) # can also do Ridge(alpha)
# model.fit(X_train, y_train)

In [ ]:
# r2_score = model.score(X_test, y_test)
# print(f"R2 on test set: {r2_score:.4f}")

In [ ]:
# from models.splice_model import _splice_cfg
# root, models = _splice_cfg()
# concepts = torch.load(os.path.join(root, models[SPLICE_MODEL]["concepts"]), weights_only=False)
# print(top_concepts(model, k=K, concepts=concepts))